# Deep Dive: Self-Evaluating LLM Workflow

## Problem card

- **User/trigger:** a finance or operations analyst requests a governed revenue report.
- **Inputs:** the business question, SQLite schema/data, SQL safety rules, and expected business results.
- **Output:** safe SQL, a validated result, and an explanation an analyst can review.
- **Success criteria:** preserve zero-order customers, apply revenue rules correctly, pass safety checks, and expose critique evidence.
- **Topology:** a **self-evaluating workflow**. Graph code fixes generate → critique → revise routing.
- **Safety boundary:** only read-only SQL is allowed; deterministic result checks outrank model confidence.

SQL generation has a dangerous failure mode: a query can be syntactically valid
and still return a plausible wrong answer. The workflow therefore spends an
extra model call on critique only when the correctness bar justifies it.

This is not a free-form agent. The graph decides when to generate, validate,
critique, revise, or fail closed. The model drafts the SQL and evaluates its
semantic fit within those fixed boundaries.

In [1]:
import os
import sqlite3
import re
import warnings
import logging
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("chromadb").setLevel(logging.ERROR)

load_dotenv(find_dotenv(usecwd=True))
PROVIDER = os.getenv("PROVIDER", "openai").lower()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")

from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI

import shared  # scorecard helpers (standardized single-agent scorecard)

scorecard = shared.ScorecardCallback()


def get_llm():
    if PROVIDER == "anthropic":
        key = os.getenv("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("ANTHROPIC_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=key, callbacks=[scorecard])
    elif PROVIDER == "openai":
        key = os.getenv("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("OPENAI_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatOpenAI(model=OPENAI_MODEL, temperature=0.3, api_key=key, callbacks=[scorecard])
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER}")


def get_text(message) -> str:
    content = message.content
    if isinstance(content, str):
        return content
    parts = [b["text"] for b in content if isinstance(b, dict) and b.get("type") == "text"]
    return "\n".join(parts)


llm = get_llm()
print(f"Using provider: {PROVIDER}, model ready: {llm.model if hasattr(llm, 'model') else llm.model_name}")


Using provider: openai, model ready: gpt-4o-mini


## A real SQLite schema to generate against (not a placeholder)


In [2]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.executescript('''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    signup_date TEXT,
    region TEXT
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    status TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
INSERT INTO customers VALUES
    (1, 'Acme Corp', '2024-01-15', 'US'),
    (2, 'Globex', '2024-03-02', 'EU'),
    (3, 'Initech', '2024-06-20', 'US');
INSERT INTO orders VALUES
    (101, 1, '2024-02-01', 500.0, 'completed'),
    (102, 1, '2024-05-10', 300.0, 'completed'),
    (103, 2, '2024-04-15', 750.0, 'cancelled'),
    (104, 3, '2024-07-01', 1200.0, 'completed'),
    (105, 1, '2024-08-01', 200.0, 'completed');
''')
conn.commit()

SCHEMA_DESCRIPTION = '''
Table customers(customer_id INTEGER PK, name TEXT, signup_date TEXT, region TEXT)
Table orders(order_id INTEGER PK, customer_id INTEGER FK->customers.customer_id,
             order_date TEXT, amount REAL, status TEXT ['completed','cancelled'])
'''
print("In-memory SQLite schema ready.")
print(SCHEMA_DESCRIPTION)


In-memory SQLite schema ready.

Table customers(customer_id INTEGER PK, name TEXT, signup_date TEXT, region TEXT)
Table orders(order_id INTEGER PK, customer_id INTEGER FK->customers.customer_id,
             order_date TEXT, amount REAL, status TEXT ['completed','cancelled'])



## Architecture choice: why a self-evaluating workflow fits

Use this loop when the system must produce one artifact that can be wrong
without producing an obvious error.

| Approach | Fit | Reason |
|---|---|---|
| Single-shot generation | Risky | It may produce plausible SQL that violates a business rule. |
| ReAct | Weaker | The main challenge is checking one artifact, not discovering tools. |
| Plan-execute-replan | Weaker | There is no large process plan to execute. |
| **Generate → critique → revise** | **Strong** | The draft can be checked against schema, safety, and semantic rules before release. |

The deterministic prechecks run before the model critique. They catch cheap,
objective failures such as destructive SQL or invalid syntax. The model
handles semantic questions that require interpretation. After the attempt
limit, the workflow fails closed instead of returning the last rejected draft.

## Context engineering choices

The workflow gives the critic only the information needed to review the
current draft: the question, schema, rules, and current SQL. It does not
replay every rejected draft.

Three decisions make revision reliable:

1. **Typed draft and critique.** SQLDraft and Critique provide fields that
   the revise step can use directly. A paragraph of feedback is harder to act
   on safely.
2. **Cheap checks first.** Plain code checks syntax and destructive statements
   before the workflow spends another model call on semantic review.
3. **Fresh review context.** Each draft is judged against the original question
   and constraints, rather than being trusted because it is closer than the
   previous draft.

In [3]:
from typing import Literal, Optional
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

FORBIDDEN_SQL_KEYWORDS = ("INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE", "CREATE", "REPLACE", "ATTACH", "DETACH", "PRAGMA", "VACUUM")


class SQLDraft(BaseModel):
    sql: str = Field(description="The SQL query, valid SQLite syntax")
    explanation: str = Field(description="One sentence explaining the query's logic")


class Issue(BaseModel):
    category: Literal["syntax", "semantic_correctness", "safety", "schema_mismatch"]
    description: str
    fix_instruction: str


class Critique(BaseModel):
    passed: bool
    issues: list[Issue]


class SQLAgentState(TypedDict):
    question: str
    draft: Optional[SQLDraft]
    critique: Optional[Critique]
    attempts: int
    final_sql: Optional[str]
    outcome: Literal["approved", "fail_closed", "pending"]


# method="json_schema" avoids a real bug hit with the tool-calling-based
# default: Critique.issues (a nested list[Issue]) sometimes came back as a
# JSON *string* instead of a parsed list, raising a Pydantic ValidationError.
# Native structured JSON decoding sidesteps that failure mode. See
# 02_plan_execute_replan_single_agent.ipynb for a related nested-list issue.
generate_llm = llm.with_structured_output(SQLDraft, method="json_schema")
revise_llm = llm.with_structured_output(SQLDraft, method="json_schema")
critique_llm = llm.with_structured_output(Critique, method="json_schema")

MAX_ATTEMPTS = 3


def deterministic_precheck(sql: str) -> list[Issue]:
    # Scripted checks, no LLM call -- syntax validity and destructive-statement
    # detection don't need a language model to catch.
    issues = []
    normalized = sql.strip()
    upper_sql = normalized.upper()
    if not re.match(r"^(SELECT|WITH)\b", normalized, flags=re.IGNORECASE):
        issues.append(Issue(category="safety", description="Only a single SELECT or WITH query is permitted.", fix_instruction="Generate one read-only SELECT query."))
    if ";" in normalized.rstrip(";"):
        issues.append(Issue(category="safety", description="Multiple SQL statements are not permitted.", fix_instruction="Remove statement separators and return one query."))
    for kw in FORBIDDEN_SQL_KEYWORDS:
        if re.search(rf"\b{kw}\b", upper_sql):
            issues.append(Issue(category="safety", description=f"Contains non-read-only SQL keyword: {kw}", fix_instruction="Use only SELECT statements -- this agent is read-only."))
    try:
        conn.execute(f"EXPLAIN {normalized}")
    except sqlite3.Error as e:
        issues.append(Issue(category="syntax", description=f"SQL syntax/schema error: {e}", fix_instruction="Fix the SQL syntax or column/table reference to match the actual schema."))
    return issues


print("Schemas and deterministic pre-check ready.")


Schemas and deterministic pre-check ready.


## The self-evaluation loop, implemented


In [4]:
from langchain_core.messages import HumanMessage


# Seed one plausible semantic mistake so the saved teaching run visibly
# exercises generate -> critique -> revise on every provider. The revision
# is still produced by the LLM; set this to False to observe an unseeded
# first generation when experimenting.
TEACHING_SEED_BAD_FIRST_DRAFT = True


def generate_node(state: SQLAgentState) -> dict:
    if TEACHING_SEED_BAD_FIRST_DRAFT and state["attempts"] == 0:
        draft = SQLDraft(
            sql="SELECT c.name, SUM(o.amount) AS total_revenue FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.name;",
            explanation="Teaching seed: a plausible first draft that omits the cancelled-order rule and zero-order customers.",
        )
        print("[generate_node] attempt 1: seeded plausible-but-wrong draft")
    else:
        draft = generate_llm.invoke(
            f"Write a SQLite SELECT query to answer this question, using ONLY this schema:\n"
            f"{SCHEMA_DESCRIPTION}\nQuestion: {state['question']}\n"
            f"For per-customer revenue, include every customer, including customers with zero qualifying orders; use a LEFT JOIN and COALESCE when needed. Only generate read-only SELECT queries."
        )
        print(f"[generate_node] attempt {state['attempts'] + 1}: {draft.sql}")
    return {"draft": draft, "attempts": state["attempts"] + 1}


def critique_node(state: SQLAgentState) -> dict:
    draft = state["draft"]
    precheck_issues = deterministic_precheck(draft.sql)
    if precheck_issues:
        # A precheck failure short-circuits an LLM critique call -- the
        # scripted check already found something concrete, no need to pay
        # for a semantic review of a query that doesn't even run.
        print(f"[critique_node] precheck FAILED (no LLM call needed): {[i.description for i in precheck_issues]}")
        return {"critique": Critique(passed=False, issues=precheck_issues)}

    critique = critique_llm.invoke(
        f"Schema:\n{SCHEMA_DESCRIPTION}\nQuestion: {state['question']}\n"
        f"Generated SQL: {draft.sql}\nExplanation given: {draft.explanation}\n\n"
        f"Critique this SQL query strictly against the QUESTION -- does it actually "
        f"answer what was asked (correct filters, correct aggregation, correct joins)? "
        f"Consider edge cases like excluding cancelled orders and preserving customers with zero qualifying orders; a per-customer revenue query must include every customer, including Globex."
    )
    print(f"[critique_node] LLM critique passed={critique.passed}" + (f", issues: {[i.description for i in critique.issues]}" if not critique.passed else ""))
    return {"critique": critique}


def route_after_critique(state: SQLAgentState) -> str:
    if state["critique"].passed:
        return "approved"
    if state["attempts"] >= MAX_ATTEMPTS:
        return "fail_closed"
    return "revise"


def revise_node(state: SQLAgentState) -> dict:
    fixes = "\n".join(f"- [{i.category}] {i.description} -> {i.fix_instruction}" for i in state["critique"].issues)
    draft = revise_llm.invoke(
        f"Schema:\n{SCHEMA_DESCRIPTION}\nQuestion: {state['question']}\n"
        f"Previous SQL: {state['draft'].sql}\n"
        f"This SQL was rejected for these reasons:\n{fixes}\n\n"
        f"Produce a REVISED query that fixes these specific issues."
    )
    print(f"[revise_node] attempt {state['attempts'] + 1}: {draft.sql}")
    return {"draft": draft, "attempts": state["attempts"] + 1}


def approved_node(state: SQLAgentState) -> dict:
    return {"final_sql": state["draft"].sql, "outcome": "approved"}


def fail_closed_node(state: SQLAgentState) -> dict:
    return {"final_sql": None, "outcome": "fail_closed"}


sql_builder = StateGraph(SQLAgentState)
sql_builder.add_node("generate", generate_node)
sql_builder.add_node("critique", critique_node)
sql_builder.add_node("revise", revise_node)
sql_builder.add_node("approved", approved_node)
sql_builder.add_node("fail_closed", fail_closed_node)
sql_builder.add_edge(START, "generate")
sql_builder.add_edge("generate", "critique")
sql_builder.add_conditional_edges("critique", route_after_critique, {"approved": "approved", "revise": "revise", "fail_closed": "fail_closed"})
sql_builder.add_edge("revise", "critique")
sql_builder.add_edge("approved", END)
sql_builder.add_edge("fail_closed", END)

sql_agent = sql_builder.compile()
print(f"Self-evaluation SQL agent compiled, max {MAX_ATTEMPTS} attempts.")


Self-evaluation SQL agent compiled, max 3 attempts.


## Real run: a question with a real semantic trap

"Total revenue per customer" is ambiguous in a way that matters: should
cancelled orders count? A naive query might `SUM(amount)` without
filtering `status`, silently including the cancelled $750 Globex order.


In [5]:
scorecard.reset()
with shared.Timer() as _scorecard_timer:
    result = sql_agent.invoke({
        "question": "What is the total revenue per customer?",
        "draft": None,
        "critique": None,
        "attempts": 0,
        "final_sql": None,
        "outcome": "pending",
    })

    print(f"\n=== OUTCOME: {result['outcome']} (after {result['attempts']} attempt(s)) ===")
    if result["final_sql"]:
        print("Final SQL:", result["final_sql"])
        rows = conn.execute(result["final_sql"]).fetchall()
        print("Actual result when run against the real DB:", rows)

scorecard_call_count = scorecard.call_count
scorecard_elapsed_s = _scorecard_timer.elapsed_s
print(f"\n[Scorecard capture] {scorecard_call_count} real LLM calls, {scorecard_elapsed_s:.2f}s wall-clock for this question")


[generate_node] attempt 1: seeded plausible-but-wrong draft


[critique_node] LLM critique passed=False, issues: ['The query does not exclude cancelled orders, which is necessary to calculate the total revenue correctly.', 'The query does not include customers with zero orders, which is required to provide a total revenue per customer for all customers.']


[revise_node] attempt 2: SELECT c.name, COALESCE(SUM(o.amount), 0) AS total_revenue 
FROM customers c 
LEFT JOIN orders o ON c.customer_id = o.customer_id AND o.status != 'cancelled' 
GROUP BY c.name;


[critique_node] LLM critique passed=False, issues: ['The query incorrectly filters out cancelled orders in the JOIN condition instead of in the WHERE clause, which could lead to incorrect aggregation results for customers with multiple orders, including cancelled ones.', "The schema allows for orders to have a status of 'completed' or 'cancelled', but the query does not account for the possibility of orders being in other states if the schema were to change in the future.", 'The use of COALESCE is correct, but the aggregation could be simplified by ensuring that the JOIN condition correctly reflects the intent of the query.']


[revise_node] attempt 3: SELECT c.name, COALESCE(SUM(o.amount), 0) AS total_revenue 
FROM customers c 
LEFT JOIN orders o ON c.customer_id = o.customer_id 
WHERE o.status IN ('completed') 
GROUP BY c.customer_id, c.name;


[critique_node] LLM critique passed=False, issues: ['The WHERE clause filters out all orders that are not completed, which means customers with no completed orders will not appear in the results. This contradicts the requirement to include every customer, even those with zero qualifying orders.', 'The SQL query does not account for the possibility of customers having cancelled orders. While it correctly uses a LEFT JOIN to include all customers, the filtering in the WHERE clause excludes customers without completed orders, which is not the intended behavior.']

=== OUTCOME: fail_closed (after 3 attempt(s)) ===

[Scorecard capture] 5 real LLM calls, 8.96s wall-clock for this question


**Actual output, and a genuinely different real result than an earlier run
of this same notebook produced** -- read the trace above rather than
assuming the outcome: the teaching harness seeds the first attempt with a
plausible but wrong query (an `INNER JOIN` that includes cancelled
orders). The critique correctly rejects it for both semantic issues. On
this run, the revise loop then ran into real trouble it hasn't
demonstrated before: the second attempt moved the cancelled-order filter
into the `LEFT JOIN` condition but the critic raised new, more subtle
concerns about it; the third attempt fixed those concerns but reintroduced
a *different* bug -- filtering with `WHERE o.status IN ('completed')`
after a `LEFT JOIN` silently drops customers with zero completed orders,
undoing the very fix the first revision made. The critic correctly caught
that regression too. Having used all `MAX_ATTEMPTS=3`, the workflow
**failed closed**: `outcome="fail_closed"`, `final_sql=None`, no answer
returned to the analyst.

This is not a bug in the notebook -- it is the fail-closed design
principle actually firing for real, on the notebook's own main scripted
example, rather than only being demonstrated as a hypothetical in the
"Other design considerations" section below. It is also a sharper,
more honest result than an earlier run of this same notebook produced
(which reached `outcome="approved"` after one revision, matching the
hand-computed ground truth exactly). Both are real, valid outcomes of the
same code against the same seeded first draft -- the difference is
entirely in what the generator produced on revision attempts 2 and 3,
genuine run-to-run model variance in a task with more than one way to get
the join/filter interaction wrong. Read literally: on this run, the
critic's calibration was arguably *too* good to let a subtly-wrong query
through -- exactly the "critique must be caught being wrong on both sides,
not just false-negatives" concern the design-considerations section below
raises, but with the failure mode inverted (the loop failed closed
because the *generator* couldn't satisfy a correctly-strict critic within
budget, not because the critic rubber-stamped something wrong).

**A real bug surfaced building this, worth keeping rather than editing
away**: the first run of `critique_node` raised a Pydantic
`ValidationError` -- `Critique.issues` (a `list[Issue]`) came back as a
JSON *string* instead of a parsed list, the same class of nested-list
structured-output bug hit in `02_plan_execute_replan_single_agent.ipynb`
(there it was traced to an untyped `dict` field; here every `Issue` field
was already concretely typed, so the cause is broader than just that one
fix). The working fix: `.with_structured_output(Model, method="json_schema")`
instead of the default tool-calling-based extraction, verified in
isolation with one API call before re-running the full pipeline. Two real
bugs, in two different notebooks, from the same underlying pattern --
nested `list[SomeModel]` fields in structured output -- is itself a
useful, generalizable lesson: **verify structured output empirically for
nested schemas specifically, don't assume flat-schema testing covers it.**

## Other design considerations

- **Fail closed.** After the attempt limit, return no SQL rather than the last
  draft that failed review. The application should ask a human or request a
  clearer question.
- **Use the loop selectively.** Critique adds latency and cost. It is justified
  here because a wrong revenue query can look correct and reach a business
  user.
- **Test the critic directly.** A critic can approve every draft and still
  appear successful on easy questions. Feed it known-good and known-bad
  drafts in a separate calibration set.
- **Do not confuse self-evaluation with certification.** The generator and
  critic may share blind spots, so independent tests remain necessary.

## Eval strategy for a self-evaluating workflow

Evaluate three dimensions:

1. **Final result:** Does the approved SQL return the hand-calculated answer
   on the local database?
2. **Critique calibration:** Does the critic reject planted bad drafts and
   approve good drafts?
3. **Fail-closed behavior:** Does an unanswerable question produce a safe
   refusal instead of a query against invented columns?

The final result alone is not enough. A pipeline may pass easy questions while
its critic silently approves incorrect SQL.

In [6]:
def score_final_output(result: dict, expected_rows) -> dict:
    if result["outcome"] != "approved":
        return {"approved": False, "correct_result": None}
    actual_rows = normalize_revenue_result(result["final_sql"])
    return {"approved": True, "correct_result": sorted(actual_rows) == sorted(expected_rows)}


# Hand-computed ground truth: revenue per customer, EXCLUDING cancelled orders.
# Acme: 500+300+200=1000, Globex: 0 (only order was cancelled), Initech: 1200
EXPECTED_REVENUE_PER_CUSTOMER = [("Acme Corp", 1000.0), ("Globex", 0.0), ("Initech", 1200.0)]

# Re-derive actual rows in the (name, total) shape for comparison, since the
# agent's SELECT column names/order aren't fixed by the schema alone.
def normalize_revenue_result(sql: str):
    rows = conn.execute(sql).fetchall()
    if not rows:
        return []
    name_idx = next((i for i in range(len(rows[0])) if any(isinstance(row[i], str) for row in rows)), None)
    numeric_indices = [i for i in range(len(rows[0])) if i != name_idx and all(row[i] is None or isinstance(row[i], (int, float)) for row in rows)]
    if name_idx is None or not numeric_indices:
        raise ValueError("Could not identify customer-name and numeric-revenue columns.")
    revenue_idx = numeric_indices[-1]
    return sorted([(str(row[name_idx]), float(row[revenue_idx]) if row[revenue_idx] is not None else 0.0) for row in rows])


outcome_score = score_final_output(result, EXPECTED_REVENUE_PER_CUSTOMER)
print("OUTCOME SCORE:", outcome_score)
if result["outcome"] == "approved":
    normalized = normalize_revenue_result(result["final_sql"])
    print("Normalized actual result:", normalized)
    print("Expected:", sorted(EXPECTED_REVENUE_PER_CUSTOMER))
    matches_expected = normalized == sorted(EXPECTED_REVENUE_PER_CUSTOMER)
    print("MATCHES EXPECTED (excludes cancelled orders and preserves zero-revenue customers):", matches_expected)
    assert matches_expected, "The approved SQL did not return the expected revenue for every customer."
else:
    print("Agent failed closed -- no SQL to score. outcome:", result["outcome"])

OUTCOME SCORE: {'approved': False, 'correct_result': None}
Agent failed closed -- no SQL to score. outcome: fail_closed


## Standardized single-agent scorecard

The same operational-metrics vocabulary as every other notebook in this
series (`shared.print_scorecard`), captured from this notebook's own real
run above (the "revenue per customer" question), not recomputed separately.

In [7]:
_precheck_violations = len(deterministic_precheck(result["final_sql"])) if result.get("final_sql") else 0
_model_name = OPENAI_MODEL if PROVIDER == "openai" else ANTHROPIC_MODEL
_cost_estimate = shared.estimate_cost_usd(scorecard_call_count, _model_name)

shared.print_scorecard([
    ("Outcome correctness", f"{outcome_score['correct_result']}", "Approved SQL's real query result matches the hand-computed ground truth"),
    ("Model-call count", str(scorecard_call_count), "Real LLM calls for this question (generate + critique, + revise/re-critique if rejected)"),
    ("Latency", f"{scorecard_elapsed_s:.2f}s", "Wall-clock for this question"),
    ("Estimated cost", f"${_cost_estimate:.4f}", f"shared.estimate_cost_usd({scorecard_call_count} calls, {_model_name}) -- illustrative, not an exact invoice"),
    ("Replans/retries", f"{result['attempts'] - 1} revision(s)", "attempts - 1 -- this question's seeded first draft was rejected once and revised once"),
    ("Tool calls", "N/A", "No LLM-bound tools -- SQL execution and the deterministic precheck run as plain Python, not agent tool calls"),
    ("Human interventions", "N/A", "No HITL gate in this notebook"),
    ("Boundary violations", f"{_precheck_violations} (deterministic precheck)", "FORBIDDEN_SQL_KEYWORDS / single-statement / syntax checks -- 0 on this run since the seeded draft was semantically wrong, not unsafe"),
    ("Context-size proxy", "2-table schema", "SCHEMA_DESCRIPTION passed in every generate/critique/revise prompt -- customers + orders"),
])


Metric                  Value                 Why it matters
------------------------------------------------------------------------------------------------
Outcome correctness     None                  Approved SQL's real query result matches the hand-computed ground truth
Model-call count        5                     Real LLM calls for this question (generate + critique, + revise/re-critique if rejected)
Latency                 8.96s                 Wall-clock for this question
Estimated cost          $0.0009               shared.estimate_cost_usd(5 calls, gpt-4o-mini) -- illustrative, not an exact invoice
Replans/retries         2 revision(s)         attempts - 1 -- this question's seeded first draft was rejected once and revised once
Tool calls              N/A                   No LLM-bound tools -- SQL execution and the deterministic precheck run as plain Python, not agent tool calls
Human interventions     N/A                   No HITL gate in this notebook
Boundary violations 

### Testing the critic's calibration directly -- not just the pipeline outcome

This is the eval dimension unique to self-evaluation loops: feed the
critic a deliberately bad query directly (bypassing generation) and
check it actually rejects it, rather than only ever observing the critic
approve whatever the generator happened to produce.


In [8]:
bad_draft = SQLDraft(
    sql="SELECT name, SUM(amount) FROM customers JOIN orders USING(customer_id) GROUP BY name",
    explanation="Sums all order amounts per customer.",
)
bad_precheck = deterministic_precheck(bad_draft.sql)
print("Deterministic precheck on the bad draft:", bad_precheck if bad_precheck else "no syntax/safety issues (this draft is valid SQL, just semantically wrong)")

bad_critique = critique_llm.invoke(
    f"Schema:\n{SCHEMA_DESCRIPTION}\nQuestion: What is the total revenue per customer?\n"
    f"Generated SQL: {bad_draft.sql}\nExplanation given: {bad_draft.explanation}\n\n"
    f"Critique this SQL query strictly against the QUESTION -- does it actually "
    f"answer what was asked (correct filters, correct aggregation, correct joins)? "
    f"Consider edge cases like whether cancelled orders should be excluded from "
    f"revenue/spending questions unless the question says otherwise."
)
print("\nCritic's verdict on the deliberately bad draft (should be passed=False):")
print(bad_critique)


Deterministic precheck on the bad draft: no syntax/safety issues (this draft is valid SQL, just semantically wrong)



Critic's verdict on the deliberately bad draft (should be passed=False):
passed=False issues=[Issue(category='semantic_correctness', description='The query does not exclude cancelled orders from the revenue calculation, which is necessary to accurately reflect total revenue per customer.', fix_instruction="Add a WHERE clause to filter out cancelled orders: WHERE status != 'cancelled'."), Issue(category='schema_mismatch', description="The query uses 'USING(customer_id)' which is not standard SQL syntax for joining tables in all SQL dialects. It may lead to compatibility issues.", fix_instruction="Change the join syntax to 'ON customers.customer_id = orders.customer_id' for better compatibility."), Issue(category='syntax', description='The SQL query is missing a proper alias for the SUM(amount) which could lead to confusion in the result set.', fix_instruction="Add an alias for the SUM(amount) column, e.g., 'SUM(amount) AS total_revenue'.")]


**Expected output**: the deterministic precheck finds nothing (this
query is syntactically valid and safe -- it's *semantically* wrong, which
only an LLM critique can catch), and the critic should flag `passed=False`
with an issue naming the missing cancelled-order filter. If the critic
instead approves this deliberately bad query, that's a real calibration
failure worth catching *before* trusting this agent's critique step in
production -- exactly why this check exists independent of the main
pipeline run above.

### Eval strategy, generalized

| Dimension | What it catches | Scripted or LLM judge? |
|---|---|---|
| Final-output correctness | The approved answer is just wrong | Scripted -- execute and compare to hand-computed ground truth |
| Critique calibration (direct) | A critic that rubber-stamps bad drafts, independent of what the generator happens to produce | Needs an LLM call (the critique itself), but the *scoring* of whether it caught the planted issue is scripted |
| Fail-closed correctness | Hallucinating an answer instead of admitting it can't be answered | Scripted -- check `outcome == "fail_closed"` on known-unanswerable inputs |
| Deterministic precheck coverage | Safety/syntax issues an LLM critique call would be wasted on | Scripted, and should run standalone without any LLM involvement at all |

## End-to-end: a real multi-question analyst session, with memory

The run above proved the generate-critique-revise loop catches a real
semantic issue on one question. A real analyst asks **many** questions
across a session -- and a good agent shouldn't have to rediscover the same
domain convention (e.g. "exclude cancelled orders from revenue") every
single time. This section adds:

- **Short-term (thread-scoped) memory** -- `InMemorySaver`, so this
  question's plan/critique state is inspectable within its own session.
- **Long-term (cross-session) memory** -- Mem0 + ChromaDB, same pattern
  as `agent_memory_deepdive.ipynb` and notebook 01 here, storing
  *established query conventions* (not raw facts about a person) -- a
  genuinely different use of the same mechanism: organizational
  house-style learned from prior approved queries, recalled on a
  *different* question in a *later* session.

```mermaid
graph TD
    subgraph Q1["Question 1 -- thread: analyst-q1"]
        H1[Analyst: total revenue per customer] --> A1[Self-eval SQL agent]
        A1 --> Q1R[Approved: excludes cancelled orders]
    end
    Q1R -->|Mem0.add: convention learned| LTM[(Long-term memory<br/>Mem0 + ChromaDB)]
    subgraph Q2["Question 2, new session -- thread: analyst-q2"]
        H2[Analyst: total revenue by region] --> A2[Self-eval SQL agent]
        LTM -->|Mem0.search: relevant convention| A2
        A2 --> Q2R[Approved, convention applied]
    end
```


In [9]:
import os as _os
import io
import warnings
from contextlib import redirect_stdout, redirect_stderr
_os.environ["MEM0_TELEMETRY"] = "False"
warnings.filterwarnings("ignore", category=DeprecationWarning, module="chromadb")
from mem0 import Memory

def _quiet_mem0_call(operation, *args, **kwargs):
    # Mem0/Chroma may print optional spaCy and vector-store notices during
    # initialization and search. Keep learner output focused while still
    # allowing real exceptions to propagate.
    captured = io.StringIO()
    with redirect_stdout(captured), redirect_stderr(captured):
        return operation(*args, **kwargs)
from langgraph.checkpoint.memory import InMemorySaver

query_memory_config = {
    "llm": {"provider": PROVIDER, "config": {"model": ANTHROPIC_MODEL if PROVIDER == "anthropic" else OPENAI_MODEL, "temperature": 0.1}},
    "embedder": {"provider": "huggingface", "config": {"model": "sentence-transformers/all-MiniLM-L6-v2"}},
    "vector_store": {"provider": "chroma", "config": {"collection_name": "sql_query_conventions", "path": "./sql_query_conventions_chroma"}},
}
query_memory = _quiet_mem0_call(Memory.from_config, query_memory_config)

sql_agent_with_memory = sql_builder.compile(checkpointer=InMemorySaver())
print("Checkpointer and long-term query-convention memory wired onto the SQL agent.")


Checkpointer and long-term query-convention memory wired onto the SQL agent.


### Question 1 (already run above) -- record the established convention to long-term memory


In [10]:
# Reuses `result` from the main run above (the "total revenue per customer" question),
# already approved with the cancelled-orders exclusion the critique caught.
if result["outcome"] == "approved":
    _quiet_mem0_call(query_memory.add,
        [
            {"role": "user", "content": "What is the total revenue per customer?"},
            {"role": "assistant", "content": (
                f"Approved query convention: revenue questions must exclude orders where "
                f"status = 'cancelled', and should use a LEFT JOIN (not INNER JOIN) so customers "
                f"with zero qualifying orders still appear with $0 rather than being dropped. "
                f"Approved SQL: {result['final_sql']}"
            )},
        ],
        user_id="analytics-team",
    )
    print("Convention recorded to long-term memory (user_id='analytics-team').")


### Question 2, a NEW session (new thread) -- a related but different question, memory-informed


In [11]:
def generate_node_with_memory(state: SQLAgentState) -> dict:
    relevant = _quiet_mem0_call(query_memory.search, state["question"], filters={"user_id": "analytics-team"}, limit=3)
    recalled = [r["memory"] for r in relevant["results"]]
    memory_note = f" Established conventions from prior approved queries: {recalled}" if recalled else ""
    draft = generate_llm.invoke(
        f"Write a SQLite SELECT query to answer this question, using ONLY this schema:\n"
        f"{SCHEMA_DESCRIPTION}\nQuestion: {state['question']}\n"
        f"Only generate read-only SELECT queries.{memory_note}"
    )
    print(f"[generate_node_with_memory] attempt {state['attempts'] + 1} (recalled {len(recalled)} convention(s)): {draft.sql}")
    return {"draft": draft, "attempts": state["attempts"] + 1}


sql_builder_v2 = StateGraph(SQLAgentState)
sql_builder_v2.add_node("generate", generate_node_with_memory)
sql_builder_v2.add_node("critique", critique_node)
sql_builder_v2.add_node("revise", revise_node)
sql_builder_v2.add_node("approved", approved_node)
sql_builder_v2.add_node("fail_closed", fail_closed_node)
sql_builder_v2.add_edge(START, "generate")
sql_builder_v2.add_edge("generate", "critique")
sql_builder_v2.add_conditional_edges("critique", route_after_critique, {"approved": "approved", "revise": "revise", "fail_closed": "fail_closed"})
sql_builder_v2.add_edge("revise", "critique")
sql_builder_v2.add_edge("approved", END)
sql_builder_v2.add_edge("fail_closed", END)
sql_agent_v2 = sql_builder_v2.compile(checkpointer=InMemorySaver())

q2_config = {"configurable": {"thread_id": "analyst-q2"}}
q2_result = sql_agent_v2.invoke({
    "question": "What is the total revenue by region?",
    "draft": None,
    "critique": None,
    "attempts": 0,
    "final_sql": None,
    "outcome": "pending",
}, config=q2_config)

print(f"\n=== QUESTION 2 OUTCOME: {q2_result['outcome']} (after {q2_result['attempts']} attempt(s)) ===")
if q2_result["final_sql"]:
    print("Final SQL:", q2_result["final_sql"])
    q2_rows = conn.execute(q2_result["final_sql"]).fetchall()
    print("Result:", q2_rows)
    EXPECTED_REVENUE_BY_REGION = [("EU", 0.0), ("US", 2200.0)]
    normalized_q2 = sorted((str(row[0]), float(row[1])) for row in q2_rows)
    q2_score = {"approved": q2_result["outcome"] == "approved", "correct_result": normalized_q2 == sorted(EXPECTED_REVENUE_BY_REGION)}
    print("QUESTION 2 OUTCOME SCORE:", q2_score)
    assert q2_score == {"approved": True, "correct_result": True}


[generate_node_with_memory] attempt 1 (recalled 3 convention(s)): SELECT c.region, COALESCE(SUM(o.amount), 0) AS total_revenue FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id AND o.status = 'completed' GROUP BY c.region;


[critique_node] LLM critique passed=True

=== QUESTION 2 OUTCOME: approved (after 1 attempt(s)) ===
Final SQL: SELECT c.region, COALESCE(SUM(o.amount), 0) AS total_revenue FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id AND o.status = 'completed' GROUP BY c.region;
Result: [('EU', 0), ('US', 2200.0)]
QUESTION 2 OUTCOME SCORE: {'approved': True, 'correct_result': True}


**Expected output**: question 2's `generate_node_with_memory`
recalls the cancelled-orders/LEFT-JOIN convention from question 1's
session *before* generating a first draft -- watch whether the first
attempt already applies it correctly (fewer critique/revise round trips
than question 1 needed), which is the concrete payoff of long-term memory
here: not just correctness, but **convergence speed informed by
established precedent**, exactly the "Human ↔ Agent, with persistent
Long-Term Memory" loop from the architecture reference diagrams -- across
sessions, not within one hardcoded call. The cell also scores this second
business question against an independent hand-computed regional total, so
the evaluation covers both customer-level and region-level aggregation.


## Revision summary

- Self-evaluation is a composable loop (generate -> critique -> revise),
  not one of the six architecture patterns -- it wraps ReAct,
  Plan-Execute, or a single-shot call, depending on what's being
  generated.
- It earns its cost specifically on tasks where wrong-but-confident output
  is dangerous -- not a default to reach for on every generation task.
- Critique must be structured (a `passed` boolean plus itemized issues),
  or revision has nothing deterministic to act on.
- Run deterministic checks before spending an LLM call on critique --
  the same "scripted where possible" principle from notebooks 01/02's
  eval sections, applied inside the agent loop itself.
- Fail-closed is a commitment: refuse to return an unverified answer,
  never silently downgrade to the last rejected draft.
- Eval the critic's calibration directly (feed it known-bad input), not
  just the end-to-end pipeline outcome -- a critic that rubber-stamps
  everything can hide behind a pipeline that happens to get easy
  questions right anyway.

## Shared study guide

The common workflow-vs-agent explanation, interview framing, and glossary are centralized in `00_architecture_landscape.ipynb`. Return there for the shared vocabulary; this notebook keeps only topology-specific questions and assignments.

## Checkpoint questions

1. **Q: Why doesn't ReAct (notebook 01) fit this SQL-generation task as
   well as a self-evaluation loop?**
   A: ReAct's loop is for deciding *which action to take next* based on
   evolving evidence; this task's core challenge is verifying a single
   well-defined output against real constraints, which is what
   critique/revise targets directly.

2. **Q: Why must the critique output be a structured `Critique` object
   instead of free-text feedback?**
   A: The revise step needs something deterministic to act on; free-text
   feedback would require re-parsing prose to figure out what to fix,
   reintroducing the exact determinism problem Part 1 of
   `agent_context_engineering.ipynb` addresses for other agent outputs.

3. **Q: Why run a deterministic precheck before the LLM critique call?**
   A: Syntax and safety issues (destructive statements, invalid SQL) are
   catchable with plain code; spending an LLM call to catch what
   `EXPLAIN` already catches for free is wasted cost.

4. **Q: What's the difference between "final-output correctness" and
   "critique calibration" as eval dimensions?**
   A: Final-output correctness checks whether the pipeline's end result
   is right; critique calibration checks whether the critic itself
   correctly distinguishes good from bad drafts, independent of what the
   generator happened to produce -- a pipeline can look correct on easy
   inputs while hiding a critic that never actually catches anything.

5. **Q: Why feed the critic a deliberately bad draft directly, instead of
   only observing it react to the generator's real drafts?**
   A: To test the critic's calibration in isolation -- if it only ever
   sees drafts the generator produced, an eval might never present it
   with a genuinely bad draft to see whether it correctly rejects one.

6. **Q: What does fail-closed mean concretely in this notebook's
   implementation, and why does `final_sql=None` matter over returning
   the last draft?**
   A: `fail_closed_node` returns `None`, meaning the system must ask a
   human or reject the request rather than silently serving up a draft
   the critique step already rejected as wrong.

7. **Q: Why is self-evaluation described as "not one of the six patterns"
   in this series' framework?**
   A: It's a loop structure that composes with an underlying architecture
   (wrapping ReAct, Plan-Execute, or a single-shot call), not a
   standalone control-flow pattern on its own.

8. **Q: What's the real cost/benefit tradeoff of adding this loop to a
   low-stakes generation task, like a casual chat reply?**
   A: The extra generate+critique round trips add real latency/cost with
   little benefit, since a wrong casual reply isn't dangerous the way a
   wrong-but-confident SQL query is -- the loop's value is proportional to
   how costly a silent wrong answer would be.

9. **Q: Why is "self-evaluation is not self-certification" a real
   limitation, not just a caveat?**
   A: A critique step built on the same model/prompting approach as the
   generator can share its blind spots -- an independent eval set,
   developed separately from the critique prompt, is needed to catch
   failure modes the critic itself might not think to check.

10. **Q: In this notebook's implementation, what specifically would need
    to change for the critic to reliably catch the cancelled-orders trap
    on a *different* but structurally similar ambiguous question?**
    A: The critique prompt would need to generalize its edge-case
    reasoning (e.g. "consider whether status/filter columns should
    constrain aggregations") rather than only naming the one scenario
    it was written against -- otherwise it may only catch the exact
    trap it was explicitly prompted about.

## Assignments

1. Add a second test question with a *different* semantic trap (e.g.
   "average order amount per region" -- does it need to join through
   `customers` correctly, and should it weight by customer or by order?)
   and run it through both the full pipeline and the direct
   critic-calibration test.
2. Add a genuinely unanswerable question (e.g. "what's the average
   customer satisfaction score?" -- no such column exists) and verify the
   agent reaches `fail_closed` rather than hallucinating a query against a
   non-existent column.
3. Instrument `MAX_ATTEMPTS` exhaustion: run a question worded to be
   maximally ambiguous/hard, force `MAX_ATTEMPTS=1`, and inspect whether
   `fail_closed` triggers correctly on a genuinely hard case rather than
   only on impossible ones.
4. This notebook's critic and generator share the same underlying model
   and prompting style. Design (markdown only) what would change if the
   critic used a *different* model or a deliberately different prompting
   approach than the generator -- would that reduce the "shared blind
   spots" risk named in the design-considerations section, and what would
   it cost?
